In [ ]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 191, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 191 (delta 75), reused 161 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (191/191), 20.28 MiB | 20.02 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [ ]:
import time
import numpy as np
import centpy
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [ ]:
class Euler2d(centpy.Equation2d):

    def pressure(self, u):
        return (self.gamma - 1.0) * (
            u[:, :, 3] - 0.5 * (u[:, :, 1] ** 2 + u[:, :, 2] ** 2) / u[:, :, 0]
        )

    def euler_data(self):
        gamma = self.gamma
        p_one, p_two, p_three, p_four = 1.5, 0.3, 0.029, 0.3

        upper_right, upper_left, lower_right, lower_left = np.ones((4, 4))

        upper_right[0] = 1.5
        upper_right[1] = 0.0
        upper_right[2] = 0.0
        upper_right[3] = (p_one / (gamma - 1.0) + 0.5 * (upper_right[1]**2 + upper_right[2]**2) / upper_right[0])

        upper_left[0] = 0.5323
        upper_left[1] = 1.206 * upper_left[0]
        upper_left[2] = 0.0
        upper_left[3] = (p_two / (gamma - 1.0) + 0.5 * (upper_left[1]**2 + upper_left[2]**2) / upper_left[0])

        lower_right[0] = 0.5323
        lower_right[1] = 0.0
        lower_right[2] = 1.206 * lower_right[0]
        lower_right[3] = (p_four / (gamma - 1.0) + 0.5 * (lower_right[1]**2 + lower_right[2]**2) / lower_right[0])

        lower_left[0] = 0.138
        lower_left[1] = 1.206 * lower_left[0]
        lower_left[2] = 1.206 * lower_left[0]
        lower_left[3] = (p_three / (gamma - 1.0) + 0.5 * (lower_left[1]**2 + lower_left[2]**2) / lower_left[0])

        return upper_right, upper_left, lower_right, lower_left

    def initial_data(self):
        u = np.empty((self.J + 4, self.K + 4, 4))
        midJ = int(self.J / 2) + 2
        midK = int(self.K / 2) + 2

        one_matrix = np.ones(u[midJ:, midK:].shape)
        upper_right, upper_left, lower_right, lower_left = self.euler_data()

        u[midJ:, midK:] = upper_right * one_matrix
        u[:midJ, midK:] = upper_left * one_matrix
        u[midJ:, :midK] = lower_right * one_matrix
        u[:midJ, :midK] = lower_left * one_matrix
        return u

    def boundary_conditions(self, u):
        upper_right, upper_left, lower_right, lower_left = self.euler_data()

        if self.odd:
            j = slice(1, -2)
            u[j, 0], u[j, -2], u[j, -1] = u[j, 1], u[j, -3], u[j, -3]
            u[0, j], u[-2, j], u[-1, j] = u[1, j], u[-3, j], u[-3, j]

            u[-2, -2], u[-1, -2], u[-2, -1], u[-1, -1] = upper_right, upper_right, upper_right, upper_right
            u[0, -2], u[0, -1] = upper_left, upper_left
            u[0, 0], u[0, 1], u[1, 0], u[1, 1] = lower_left, lower_left, lower_left, lower_left
            u[-2, 0], u[-1, 0], u[-2, 1], u[-1, 1] = lower_right, lower_right, lower_right, lower_right
        else:
            j = slice(2, -1)
            u[j, 0], u[j, 1], u[j, -1] = u[j, 2], u[j, 2], u[j, -2]
            u[0, j], u[1, j], u[-1, j] = u[2, j], u[2, j], u[-2, j]

            u[-1, -2], u[-1, -1] = upper_right, upper_right
            u[0, -2], u[0, -1], u[1, -2], u[1, -1] = upper_left, upper_left, upper_left, upper_left
            u[0, 0], u[0, 1], u[1, 0], u[1, 1] = lower_left, lower_left, lower_left, lower_left
            u[-1, 0], u[-1, 1] = lower_right, lower_right

    def flux_x(self, u):
        f = np.empty_like(u)
        p = self.pressure(u)
        f[:, :, 0] = u[:, :, 1]
        f[:, :, 1] = u[:, :, 1] ** 2 / u[:, :, 0] + p
        f[:, :, 2] = u[:, :, 1] * u[:, :, 2] / u[:, :, 0]
        f[:, :, 3] = (u[:, :, 3] + p) * u[:, :, 1] / u[:, :, 0]
        return f

    def flux_y(self, u):
        g = np.empty_like(u)
        p = self.pressure(u)
        g[:, :, 0] = u[:, :, 2]
        g[:, :, 1] = u[:, :, 1] * u[:, :, 2] / u[:, :, 0]
        g[:, :, 2] = u[:, :, 2] ** 2 / u[:, :, 0] + p
        g[:, :, 3] = (u[:, :, 3] + p) * u[:, :, 2] / u[:, :, 0]
        return g

    def spectral_radius_x(self, u):
        # Если массив 204x204 (имеет теневые ячейки) - берем только центр 200x200.
        # Если массив 200x200 (интерфейсы) - оставляем как есть.
        if u.shape[0] == self.J + 4:
            j0 = slice(2, -2)
            u_core = u[j0, j0]
        else:
            u_core = u

        rho = u_core[..., 0]
        vx = u_core[..., 1] / rho
        vy = u_core[..., 2] / rho
        # Вычисление давления: p = (gamma - 1) * (E - 0.5 * rho * (v_x^2 + v_y^2))
        p = (self.gamma - 1.0) * (u_core[..., 3] - 0.5 * rho * (vx ** 2 + vy ** 2))

        # Защита от отрицательного давления из-за численных погрешностей
        p = np.maximum(p, 1e-10)
        rho = np.maximum(rho, 1e-10)

        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vx) + c

    def spectral_radius_y(self, u):
        if u.shape[0] == self.J + 4:
            j0 = slice(2, -2)
            u_core = u[j0, j0]
        else:
            u_core = u

        rho = u_core[..., 0]
        vx = u_core[..., 1] / rho
        vy = u_core[..., 2] / rho
        p = (self.gamma - 1.0) * (u_core[..., 3] - 0.5 * rho * (vx ** 2 + vy ** 2))

        # Защита от отрицательного давления/плотности
        p = np.maximum(p, 1e-10)
        rho = np.maximum(rho, 1e-10)

        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vy) + c


def run_cpu_benchmark():
    # Параметры из оригинального примера Euler_2d
    pars = centpy.Pars2d(
        x_init=0., x_final=1.,
        y_init=0., y_final=1.,
        J=200, K=200,
        t_final=0.4,
        dt_out=0.005,
        cfl=0.475,
        scheme="sd2"
    )
    pars.gamma = 1.4

    eqn = Euler2d(pars)
    solver = centpy.Solver2d(eqn)

    print("--- Запуск CPU Бенчмарка (Euler 2D, сетка 200x200) ---")

    t0 = time.time()
    solver.solve()
    t1 = time.time()

    print(f"\n[CPU centpy] Полное время выполнения Эйлера: {t1 - t0:.4f} секунд")

    return solver

if __name__ == "__main__":
    soln = run_cpu_benchmark()

--- Запуск CPU Бенчмарка (Euler 2D, сетка 200x200) ---

[CPU centpy] Полное время выполнения Эйлера: 62.2358 секунд


In [ ]:
# Animation
fig, ax = plt.subplots()
ax.set_xlim(soln.x_init, soln.x_final)
ax.set_ylim(soln.y_init, soln.y_final)

# Извлекаем сетку координат и начальные данные для первого кадра
x_grid = soln.x[1:-1]
y_grid = soln.y[1:-1]
data_init = soln.u_n[0, 1:-1, 1:-1, 0]

# 1. Создаем фоновую тепловую карту с помощью imshow
# Параметр extent привязывает матрицу к реальным координатам
im = ax.imshow(
    data_init,
    extent=[soln.x_init, soln.x_final, soln.y_init, soln.y_final],
    origin='lower',             # Гарантирует, что ось Y направлена вверх
    cmap='coolwarm',             # Или 'magma', 'viridis', 'turbo'
    interpolation='bicubic',    # Сглаживание для профессионального вида
    aspect='auto'               # Позволяет осям масштабироваться независимо
)

# Добавляем цветовую шкалу для наглядности (опционально, но рекомендуется)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Значение u_n')

# 2. Отрисовываем начальные контуры поверх тепловой карты
ax.contour(
    x_grid, y_grid, data_init,
    levels=20,
    colors='black',
    alpha=0.5,
    linewidths=0.5
)

# Функция обновления для каждого кадра анимации
def animate(i):
    # Получаем данные текущего шага
    data = soln.u_n[i, 1:-1, 1:-1, 0]

    # Быстрое обновление тепловой карты (работает быстрее, чем перерисовка)
    im.set_data(data)

    # Если глобальный минимум и максимум меняются со временем,
    # можно раскомментировать следующую строку для динамической шкалы:
    # im.set_clim(vmin=data.min(), vmax=data.max())

    # Удаляем старые линии контуров из коллекции осей
    for c in ax.collections:
        c.remove()

    # Рисуем новые контурные линии для текущего кадра
    ax.contour(
        x_grid, y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    return [im]

plt.close() # Закрываем статичную фигуру, чтобы она не дублировалась в выводе

# Создаем анимацию
anim = animation.FuncAnimation(fig, animate, frames=soln.Nt, interval=100, blit=False)

# Выводим как HTML5 видео
HTML(anim.to_html5_video())